In [25]:
import pandas as pd
import os

In [26]:
train_path = "../data/train_115-00000-of-00001.parquet"
val1_path = "../data/validation-00000-of-00002.parquet"
val2_path = "../data/validation-00001-of-00002.parquet"

In [27]:
train = pd.read_parquet(train_path)
val1 = pd.read_parquet(val1_path)
val2 = pd.read_parquet(val2_path)

print("Train:", train.shape)
print("Validation 1:", val1.shape)
print("Validation 2:", val2.shape)

Train: (115, 26)
Validation 1: (1017, 26)
Validation 2: (1016, 26)


In [28]:
print(train.columns.tolist())

['id', 'locale', 'partition', 'scenario', 'scenario_str', 'intent_idx', 'intent_str', 'utt', 'annot_utt', 'worker_id', 'slot_method', 'judgments', 'tokens', 'labels', 'audio', 'path', 'is_transcript_reported', 'is_validated', 'speaker_id', 'speaker_sex', 'speaker_age', 'speaker_ethnicity_simple', 'speaker_country_of_birth', 'speaker_country_of_residence', 'speaker_nationality', 'speaker_first_language']


In [29]:
train["original_split"] = "train"
val1["original_split"] = "validation"
val2["original_split"] = "validation"

In [30]:
dataset = pd.concat(
    [train, val1, val2],
    ignore_index=True
)

print(dataset.shape)
print(dataset["original_split"].value_counts())
print("Số speaker:", dataset["speaker_id"].nunique())


(2148, 27)
original_split
validation    2033
train          115
Name: count, dtype: int64
Số speaker: 35


In [31]:
inventory = dataset[[
    "id",
    "path",
    "original_split",
    "speaker_id",
    "utt",
    "intent_str",
    "is_validated"
]].copy()

inventory.columns = [
    "audio_id",
    "audio_path",
    "original_split",
    "speaker_id",
    "transcript",
    "intent",
    "is_valid"
]
print(inventory.columns.tolist())

['audio_id', 'audio_path', 'original_split', 'speaker_id', 'transcript', 'intent', 'is_valid']


In [32]:
print(inventory.head())
print(inventory.shape)
print(inventory.isnull().sum())

  audio_id                                      audio_path original_split  \
0     9702  train-115/4dcf89cc7708ffe6339d97afd4da24f5.wav          train   
1     9671  train-115/2d48e259c29bdbf2039edfddad79cb61.wav          train   
2    10249  train-115/81da41ae4fbf09485ad6a4214439f9d0.wav          train   
3     3854  train-115/2866444482800feff8590246b56cd245.wav          train   
4    12053  train-115/ddd9fc1eee9336973c9de546c7af4794.wav          train   

                 speaker_id  \
0  657c8d982832af573ef2c039   
1  5cf03d69b094d700013e4d54   
2  5e25be7c5514e680ef436338   
3  657c8d982832af573ef2c039   
4  659ea35db097ea3c414b04c0   

                                          transcript                 intent  \
0     tôi muốn nghe một quyển sách bởi la quán trung         play_audiobook   
1  bắt đầu phát tam quốc diễn nghĩa ở chỗ mà tôi ...         play_audiobook   
2                         hãy chơi một ván trivia              play_game   
3                                 

In [33]:
print("Duplicate audio_id:",
      inventory["audio_id"].duplicated().sum())
print("Number of speakers:",
      inventory["speaker_id"].nunique())
print(
    inventory["original_split"].value_counts()
)

Duplicate audio_id: 0
Number of speakers: 35
original_split
validation    2033
train          115
Name: count, dtype: int64


In [34]:
os.makedirs("../data/metadata", exist_ok=True)

output_path = "../data/metadata/data_inventory.csv"

inventory.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_path)

Saved: ../data/metadata/data_inventory.csv


In [35]:
check = pd.read_csv(output_path)

print(check.shape)
print(check.columns.tolist())
print(check.head())

(2148, 7)
['audio_id', 'audio_path', 'original_split', 'speaker_id', 'transcript', 'intent', 'is_valid']
   audio_id                                      audio_path original_split  \
0      9702  train-115/4dcf89cc7708ffe6339d97afd4da24f5.wav          train   
1      9671  train-115/2d48e259c29bdbf2039edfddad79cb61.wav          train   
2     10249  train-115/81da41ae4fbf09485ad6a4214439f9d0.wav          train   
3      3854  train-115/2866444482800feff8590246b56cd245.wav          train   
4     12053  train-115/ddd9fc1eee9336973c9de546c7af4794.wav          train   

                 speaker_id  \
0  657c8d982832af573ef2c039   
1  5cf03d69b094d700013e4d54   
2  5e25be7c5514e680ef436338   
3  657c8d982832af573ef2c039   
4  659ea35db097ea3c414b04c0   

                                          transcript                 intent  \
0     tôi muốn nghe một quyển sách bởi la quán trung         play_audiobook   
1  bắt đầu phát tam quốc diễn nghĩa ở chỗ mà tôi ...         play_audiobook   
2 